In [ ]:
import numpy as np
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('./data.csv')
df.head()

In [ ]:
df = df.rename(columns={
    "Carimbo de data/hora": "timestamp",
    "Com qual gênero você se identifica?": "gender",
    "Qual sua idade?": "age",
    "Em qual etapa acadêmica você se encontra atualmente? ": "academic_stage",
    "Com que frequência você pratica atividade física em uma semana comum?  ": "weekly_sessions",
    "Em média, quanto tempo dura cada sessão de atividade física?": "minutes_per_session",
    "Tenho dificuldade em me concentrar nos estudos e/ou em atividades que necessitam de foco.": "difficulty_concentrating",
    "Me sinto sempre cansado, com pouca energia e disposição.": "always_tired",
    "Eu tenho uma boa qualidade de sono.": "good_sleep_quality",
    "Me sinto frequentemente para baixo, depressivo, sem esperança. Estou sempre preocupado, como se algo ruim fosse acontecer.": "often_down",
    "A falta de tempo por causa das demandas da faculdade é um empecilho para a minha prática de atividades físicas.": "lack_of_time",
    "O cansaço físico e mental decorrente dos estudos é um empecilho para a minha prática de atividades físicas.": "study_fatigue",
    "O custo financeiro (mensalidade de academia, equipamentos, etc.) é um empecilho para a minha prática de atividades físicas.": "financial_cost",
    "Eu pratico ou gostaria de praticar atividades físicas para cuidar da saúde mental e aliviar o estresse.": "physical_for_mental_health",
    "Eu pratico ou gostaria de praticar atividades físicas para melhorar a minha saúde física.": "physical_for_physical_health",
    "Eu pratico ou gostaria de praticar atividades físicas para melhorar a minha aparência.": "physical_for_appearance",
    "Eu pratico ou gostaria de praticar atividades físicas para ter um momento de lazer e/ou socialização.": "physical_for_leisure",
})

In [ ]:
columns_likert = [
    "difficulty_concentrating",
    "always_tired",
    "good_sleep_quality",
    "often_down",
    "lack_of_time",
    "study_fatigue",
    "financial_cost",
    "physical_for_mental_health",
    "physical_for_physical_health",
    "physical_for_appearance",
    "physical_for_leisure",
]

In [ ]:
weekly_sessions_map = {
    'Nunca ou quase nunca': 0,
    '1 vez por semana': 1,
    '2 a 3 vezes por semana': 2.5,
    '4 a 5 vezes por semana': 4.5,
    '6 ou mais vezes por semana': 6
}

minutes_per_session_map = {
    'Menos de 30 minutos': 15,
    '30 a 59 minutos': 45,
    '60 a 89 minutos': 75,
    '90 minutos ou mais': 90
}

likert_map = {
    'Discordo totalmente': 1,
    'Discordo': 2,
    'Neutro': 3,
    'Concordo': 4,
    'Concordo totalmente': 5,
    'Concordo Totalmente': 5
}

df = df.replace(likert_map).replace(weekly_sessions_map).replace(minutes_per_session_map)
df["weekly_minutes"] = df["weekly_sessions"] * df["minutes_per_session"]

In [ ]:
answers_order = [
    "Discordo totalmente", "Discordo", "Neutro", "Concordo", "Concordo totalmente"
]

levels_order = ["Inactive", "Slightly Active", "Active", "Highly Active"]

In [ ]:
def classify_activity(x):
    if x == 0:
        return "Inactive"
    elif x < 150:
        return "Slightly Active"
    elif x < 300:
        return "Active"
    else:
        return "Highly Active"

df["activity_level"] = df["weekly_minutes"].apply(classify_activity)

## Análise geral


In [ ]:
# Contagem absoluta
print(df["gender"].value_counts())
print(df["academic_stage"].value_counts())
print(df["age"].value_counts())


In [ ]:
df_by_level = df.groupby("activity_level")
df_by_level.describe()

In [ ]:
chart_levels_distribution = alt.Chart(df).mark_bar().encode(
    x=alt.X('activity_level:N', sort=levels_order, title='Activity level', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('count()', title='Number of respondents'),
    color=alt.Color('activity_level:N', legend=alt.Legend(title='Activity Level'), sort=levels_order),
    tooltip=['activity_level:N', 'count()']
).properties(
    title='Distribution of Activity Levels among Respondents',
    width=500
)
chart_levels_distribution


In [ ]:
df_filtered = df[df['gender'] != 'Prefiro não dizer']

gender_map = {
    "Masculino": "Male",
    "Feminino": "Female"
}
df_filtered["gender_en"] = df_filtered["gender"].map(gender_map)

# calcular proporção dentro de cada gênero
prop = (
    df_filtered.groupby(['gender_en', 'activity_level'])
    .size()
    .reset_index(name='count')
)
prop['proportion'] = prop.groupby('gender_en')['count'].transform(lambda x: x / x.sum())

# gráfico
chart_prop = alt.Chart(prop).mark_bar().encode(
    x=alt.X('activity_level:N', sort=levels_order, title='Activity Level', axis=alt.Axis(labelAngle=0)),
    xOffset='gender_en:N',
    y=alt.Y('proportion:Q', title='Proportion within each gender', axis=alt.Axis(format='%')),
    color=alt.Color('gender_en:N', legend=alt.Legend(title='Gender')),
    tooltip=['gender_en:N', 'activity_level:N', alt.Tooltip('proportion:Q', format='.1%')]
).properties(
    title='Activity Level Proportion within each Gender',
    width=500,
    height=350
)

chart_prop

In [ ]:
combined = alt.hconcat(chart_levels_distribution, chart_prop).resolve_scale(
    y='independent',    # escalas y separadas
    color='independent' # legendas separadas
)
combined

In [ ]:
df_mean = df.groupby("activity_level")[columns_likert].mean().reset_index()

# Transformar em formato longo
df_long = df_mean.melt(id_vars="activity_level", var_name="variavel", value_name="media")

heatmap = alt.Chart(df_long).mark_rect().encode(
    x=alt.X('variavel:N', title='Variável', sort=columns_likert),
    y=alt.Y('activity_level:N', sort=levels_order),
    color=alt.Color('media:Q', scale=alt.Scale(scheme='blues')),
    tooltip=['activity_level:N', 'variavel:N', 'media:Q']
).properties(
    title='Médias por variável e nível de atividade',
    width=700
)

heatmap

### Saúde mental


In [ ]:
mental_health_columns = [
    "difficulty_concentrating",
    "always_tired",
    "good_sleep_quality",
    "often_down",
]

In [ ]:
charts = []
for col in mental_health_columns:
    c = (
        alt.Chart(df)
        .mark_boxplot()
        .encode(
            x=alt.X("activity_level:N", sort=levels_order, title=None),
            y=alt.Y(f"{col}:Q", title=col.replace("_", " ").capitalize()),
            color="activity_level:N"
        )
        .properties(width=150, height=250)
    )
    charts.append(c)

boxplots = alt.hconcat(*charts).resolve_scale(y='independent').properties(
    title="Distribuição das variáveis de saúde mental por nível de atividade"
)

boxplots

In [ ]:
summary = df.groupby("activity_level")[mental_health_columns].agg(['mean','std'])
summary

In [ ]:
corr = df[["weekly_minutes"] + mental_health_columns].corr().loc["weekly_minutes", mental_health_columns]
corr

In [ ]:
mh_map = {
    "difficulty_concentrating": "Concentration Problems",
    "always_tired": "Low Energy/Fatigue",
    "good_sleep_quality": "Sleep Quality",
    "often_down": "Low Mood/Anxiety",
}
df_mh_heatmap = df.rename(columns=mh_map)

parsed_mental_health_columns = [mh_map[col] for col in mental_health_columns]

df_mh_mean = df_mh_heatmap.groupby("activity_level")[parsed_mental_health_columns].mean().reset_index()

# Transformar em formato longo
df_mh_long = df_mh_mean.melt(id_vars="activity_level", var_name="variavel", value_name="media")

heatmap = alt.Chart(df_mh_long).mark_rect().encode(
    x=alt.X('variavel:N', title='Mental Health Statement', axis=alt.Axis(labelAngle=0), sort=columns_likert),
    y=alt.Y('activity_level:N', title='Activity Level', sort=levels_order),
    color=alt.Color('media:Q', scale=alt.Scale(scheme='blues'), legend=alt.Legend(title='Mean')),
    tooltip=['activity_level:N', 'variavel:N', 'media:Q']
).properties(
    title='Mental Health Statements Agreement Mean by Activity Level',
    width=700
)

heatmap

In [ ]:
df_mh = df.rename(columns=mh_map).copy()

gender_map = {
    "Feminino": "Female",
    "Masculino": "Male",
    "Prefiro não dizer": "Prefer not to say"
}
df_mh["gender"] = df_mh["gender"].map(gender_map).fillna(df_mh["gender"])

df_mh_mean_gender = (
    df_mh.groupby(["activity_level", "gender"], dropna=False)[parsed_mental_health_columns]
    .mean()
    .reset_index()
)

df_mh_long_gender = df_mh_mean_gender.melt(
    id_vars=["activity_level", "gender"],
    var_name="Mental Health Aspect",
    value_name="Mean"
).dropna(subset=["activity_level"])

base = (
    alt.Chart(df_mh_long_gender)
    .mark_line(point=True)
    .encode(
        x=alt.X("activity_level:N", title="Activity Level", axis=alt.Axis(labelAngle=0), sort=levels_order),
        y=alt.Y("Mean:Q", title="Mean Agreement (1–5)"),
        color=alt.Color("gender:N", title="Gender"),
        tooltip=["activity_level:N", "gender:N", "Mental Health Aspect:N", alt.Tooltip("Mean:Q", format=".2f")]
    )
    .properties(width=220, height=220)
)

chart_mh_lines = base.facet(
    facet=alt.Facet("Mental Health Aspect:N", title=None, header=alt.Header(labelAngle=0, labelAlign="center")),
    columns=2  # define 2 facets por linha
).properties(
    title="Mental Health Statements Agreement Mean by Activity Level and Gender"
).resolve_scale(x="independent")

chart_mh_lines

In [ ]:
# Garantir tipo numérico e remover linhas vazias
df_mh["weekly_sessions"] = pd.to_numeric(df_mh["weekly_sessions"], errors="coerce")
df_mh_valid = df_mh.dropna(subset=["weekly_sessions", "gender"])

# Se a variável for discreta (categorias 0–6), trata como categórica ordenada
sessions_order = sorted(df_mh_valid["weekly_sessions"].unique())
df_mh_valid["weekly_sessions"] = pd.Categorical(df_mh_valid["weekly_sessions"], categories=sessions_order, ordered=True)

# Calcular médias por gênero e frequência
df_mh_mean_freq = (
    df_mh_valid.groupby(["weekly_sessions", "gender"], dropna=False)[parsed_mental_health_columns]
    .mean()
    .reset_index()
)

# Transformar em formato longo
df_mh_long_freq = df_mh_mean_freq.melt(
    id_vars=["weekly_sessions", "gender"],
    var_name="Mental Health Aspect",
    value_name="Mean"
)

# Criar gráfico facetado
chart_mh_lines_freq = (
    alt.Chart(df_mh_long_freq)
    .mark_line(point=True)
    .encode(
        x=alt.X("weekly_sessions:N", title="Weekly Sessions", sort=sessions_order),
        y=alt.Y("Mean:Q", title="Mean Agreement (1–5)"),
        color=alt.Color("gender:N", title="Gender"),
        strokeDash=alt.StrokeDash("gender:N", title="Gender"),
        tooltip=[
            "weekly_sessions:N",
            "gender:N",
            "Mental Health Aspect:N",
            alt.Tooltip("Mean:Q", format=".2f")
        ]
    )
    .properties(width=220, height=220)
)

chart_mh_lines_freq_facet = chart_mh_lines_freq.facet(
    column=alt.Column(
        "Mental Health Aspect:N",
        title=None,
        header=alt.Header(labelAngle=0, labelAlign="center")
    ),
    title="Mental Health Aspects by Weekly Sessions and Gender"
).resolve_scale(
    y='independent'
)

chart_mh_lines_freq_facet


### Barreiras


In [ ]:
barriers_columns = [
    "lack_of_time",
    "study_fatigue",
    "financial_cost"
]

In [ ]:
# formato longo
df_long = df.melt(
    value_vars=barriers_columns,
    var_name="barrier",
    value_name="answer"
).copy()

# garantir coluna numérica para order
df_long["answer_num"] = pd.to_numeric(df_long["answer"], errors="coerce")

# mapa num -> label
answer_labels = {
    1: "Discordo totalmente",
    2: "Discordo",
    3: "Neutro",
    4: "Concordo",
    5: "Concordo totalmente"
}

# coluna de rótulos para legenda
df_long["answer_label"] = df_long["answer_num"].map(answer_labels).astype(str)

# definir ordem categórica explícita para a legenda
labels_order = ["Discordo totalmente", "Discordo", "Neutro", "Concordo", "Concordo totalmente"]
df_long["answer_label"] = pd.Categorical(df_long["answer_label"], categories=labels_order, ordered=True)

# contar proporções
prop = (
    df_long.groupby(["barrier", "answer_num", "answer_label"])
    .size()
    .reset_index(name="count")
)
prop["proportion"] = prop.groupby("barrier")["count"].transform(lambda x: x / x.sum())
prop["proportion"] = prop["proportion"].astype(float)

# gráfico: color pela label (texto), order pelo número (mantém sequência correta)
chart = alt.Chart(prop).mark_bar().encode(
    x=alt.X("barrier:N", title="Barreira"),
    y=alt.Y("proportion:Q", title="Proporção"),
    color=alt.Color("answer_label:N", title="Nível de concordância", sort=labels_order),
    order=alt.Order("answer_num:Q", sort="ascending"),
    tooltip=["barrier", "answer_label", alt.Tooltip("proportion:Q", format=".2f")]
).properties(
    width=500,
    height=320,
    title="Distribuição de respostas para barreiras"
)

chart


### Motivações


In [ ]:
motivations_columns = [
    "physical_for_mental_health",
    "physical_for_physical_health",
    "physical_for_appearance",
    "physical_for_leisure"
]

In [ ]:
df_long_mot = df.melt(
    id_vars=["gender"],
    value_vars=motivations_columns,
    var_name="motivation",
    value_name="answer"
).copy()
df_long_mot["answer_num"] = pd.to_numeric(df_long_mot["answer"], errors="coerce")
df_long_mot["answer_label"] = df_long_mot["answer_num"].map(answer_labels).astype(str)
labels_order = ["Discordo totalmente","Discordo","Neutro","Concordo","Concordo totalmente"]
df_long_mot["answer_label"] = pd.Categorical(df_long_mot["answer_label"], categories=labels_order, ordered=True)

prop_general = (
    df_long_mot
    .groupby(["motivation","answer_num","answer_label"], dropna=False)
    .size()
    .reset_index(name="count")
)
prop_general["proportion"] = prop_general.groupby("motivation")["count"].transform(lambda x: x / x.sum())
prop_general["proportion"] = prop_general["proportion"].astype(float)

# verificar soma = 1 por motivation
print(prop_general.groupby("motivation")["proportion"].sum())

# gráfico geral (empilhado por resposta)
chart_general = alt.Chart(prop_general).mark_bar().encode(
    x=alt.X("motivation:N", title="Motivação"),
    y=alt.Y("proportion:Q", title="Proporção"),
    color=alt.Color("answer_label:N", title="Nível de concordância", sort=labels_order),
    order=alt.Order("answer_num:Q", sort="ascending"),
    tooltip=["motivation", "answer_label", alt.Tooltip("proportion:Q", format=".2f")]
).properties(width=500, height=320, title="Distribuição geral das motivações")

chart_general

In [ ]:
answer_labels = {
    1: "Strongly Disagree",
    2: "Disagree",
    3: "Neutral",
    4: "Agree",
    5: "Strongly Agree"
}
labels_order = list(answer_labels.values())

barriers_columns = ["financial_cost", "lack_of_time", "study_fatigue"]
motivations_columns = [
    "physical_for_appearance",
    "physical_for_leisure",
    "physical_for_mental_health",
    "physical_for_physical_health"
]

barriers_rename = {
    "financial_cost": "Financial Cost",
    "lack_of_time": "Lack of Time",
    "study_fatigue": "Tiredness (Studies)"
}

motivations_rename = {
    "physical_for_appearance": "Appearance",
    "physical_for_leisure": "Leisure/Social",
    "physical_for_mental_health": "Mental Health",
    "physical_for_physical_health": "Physical Health"
}

df_long_bar = df.melt(
    value_vars=barriers_columns,
    var_name="barrier",
    value_name="answer"
).copy()

df_long_bar["barrier"] = df_long_bar["barrier"].replace(barriers_rename)
df_long_bar["answer_num"] = pd.to_numeric(df_long_bar["answer"], errors="coerce")
df_long_bar["answer_label"] = df_long_bar["answer_num"].map(answer_labels)

bar_prop = (
    df_long_bar.groupby(["barrier", "answer_num", "answer_label"])
    .size()
    .reset_index(name="count")
)
bar_prop["proportion"] = bar_prop.groupby("barrier")["count"].transform(lambda x: x / x.sum())

chart_bar = alt.Chart(bar_prop).mark_bar().encode(
    x=alt.X("barrier:N", title="Barrier", axis=alt.Axis(labelAngle=0)),  # 👈 rótulos horizontais
    y=alt.Y("proportion:Q", title="Proportion"),
    color=alt.Color("answer_label:N", title="Agreement Level", sort=labels_order),
    order=alt.Order("answer_num:Q", sort="ascending"),
    tooltip=["barrier", "answer_label", alt.Tooltip("proportion:Q", format=".2f")]
).properties(
    width=400,
    height=320,
    title="Distribution of Responses for Barriers"
)

df_long_mot = df.melt(
    value_vars=motivations_columns,
    var_name="motivation",
    value_name="answer"
).copy()

df_long_mot["motivation"] = df_long_mot["motivation"].replace(motivations_rename)
df_long_mot["answer_num"] = pd.to_numeric(df_long_mot["answer"], errors="coerce")
df_long_mot["answer_label"] = df_long_mot["answer_num"].map(answer_labels)

mot_prop = (
    df_long_mot.groupby(["motivation", "answer_num", "answer_label"])
    .size()
    .reset_index(name="count")
)
mot_prop["proportion"] = mot_prop.groupby("motivation")["count"].transform(lambda x: x / x.sum())

chart_mot = alt.Chart(mot_prop).mark_bar().encode(
    x=alt.X("motivation:N", title="Motivation", axis=alt.Axis(labelAngle=0)),  # 👈 rótulos horizontais
    y=alt.Y("proportion:Q", title="Proportion"),
    color=alt.Color("answer_label:N", title="Agreement Level", sort=labels_order),
    order=alt.Order("answer_num:Q", sort="ascending"),
    tooltip=["motivation", "answer_label", alt.Tooltip("proportion:Q", format=".2f")]
).properties(
    width=400,
    height=320,
    title="Distribution of Responses for Motivations"
)

combined_chart = (chart_bar | chart_mot).resolve_scale(color='shared')
combined_chart


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

answer_map = {
    1: "Strongly Disagree",
    2: "Disagree",
    3: "Neutral",
    4: "Agree",
    5: "Strongly Agree"
}

likert_order = ["Strongly Disagree", "Disagree", "Neutral", "Agree", "Strongly Agree"]

barriers_columns = {
    "financial_cost": "Financial Cost",
    "lack_of_time": "Lack of Time",
    "study_fatigue": "Tiredness (Studies)"
}

motivations_columns = {
    "physical_for_appearance": "Appearance",
    "physical_for_leisure": "Leisure/Social",
    "physical_for_mental_health": "Mental Health",
    "physical_for_physical_health": "Physical Health"
}

def grouped_bars(df_subset, columns_map, title):
    # melt
    df_long = df_subset.melt(var_name="item", value_name="answer")
    df_long["item"] = df_long["item"].replace(columns_map)
    df_long["answer_label"] = df_long["answer"].map(answer_map)

    # count proportions
    counts = (
        df_long.groupby(["item", "answer_label"])
        .size()
        .unstack(fill_value=0)
    )

    # order columns
    counts = counts[likert_order]

    # proportions
    propor = counts.div(counts.sum(axis=1), axis=0)

    # plotting
    x = np.arange(len(propor.index))
    width = 0.15

    fig, ax = plt.subplots(figsize=(12, 5))

    for i, c in enumerate(propor.columns):
        ax.bar(x + i * width, propor[c], width, label=c)

    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(propor.index, rotation=0)
    ax.set_ylabel("Proportion")
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')

    plt.tight_layout()
    plt.show()

grouped_bars(df[list(barriers_columns.keys())], barriers_columns, "Distribution of Responses for Barriers")
grouped_bars(df[list(motivations_columns.keys())], motivations_columns, "Distribution of Responses for Motivations")


In [ ]:
chart_gender = alt.Chart(prop).mark_bar().encode(
    x=alt.X("motivation:N", title="Motivação"),
    y=alt.Y("proportion:Q", title="Proporção"),
    color=alt.Color("answer_label:N", title="Nível de concordância", sort=labels_order),
    order=alt.Order("answer_num:Q", sort="ascending"),
    tooltip=["motivation", "gender", "answer_label", alt.Tooltip("proportion:Q", format=".2f")]
).properties(
    width=250,
    height=300
).facet(
    column=alt.Column("gender:N", title="Gênero")
).resolve_scale(
    y='independent'
)

chart_gender